In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/genai/spark_lab"


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronzeSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

In [0]:
bronze_df = spark.read.load(f'/Volumes/{catalog_name}/genai/spark_lab/*_edited.csv', format='csv', schema=bronzeSchema)
display(bronze_df.limit(100))

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/genai/spark_lab/*_edited.csv', format='csv')
display(df.limit(100))

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronzeSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

In [0]:
bronze_df = spark.read.load(f'/Volumes/{catalog_name}/genai/spark_lab/*_edited.csv', format='csv', schema=bronzeSchema)
display(bronze_df.limit(100))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dea_course_ws.Bronze;

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
bronzeDeltaTablePath = f'/Volumes/{catalog_name}/genai/spark_lab/delta/bronze_sales_orders'
bronze_df.write.format('delta').mode('overwrite').save(bronzeDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
bronze_df.write.format('delta').mode('overwrite').saveAsTable('Bronze.bronze_sales_orders')

In [0]:
silver_df = spark.read.format("delta").table("Bronze.bronze_sales_orders")
silver_df.show(10)

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.dropDuplicates()
silver_df = silver_df.withColumn('Tax', col('UnitPrice') * 0.08)
silver_df = silver_df.withColumn('Tax', col('Tax').cast("float"))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dea_course_ws.Silver;

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
silverDeltaTablePath = f'/Volumes/{catalog_name}/genai/spark_lab/delta/silver_sales_orders'
silver_df.write.format('delta').mode('overwrite').save(silverDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
silver_df.write.format('delta').saveAsTable('Silver.silver_sales_orders')

In [0]:
gold_df = spark.read.format("delta").table("Silver.silver_sales_orders")
gold_df.show(10)

In [0]:
yearlySales = gold_df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dea_course_ws.Gold;

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
goldDeltaTablePath = f'/Volumes/{catalog_name}/genai/spark_lab/delta/gold_sales_orders'
yearlySales.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
yearlySales.write.format('delta').saveAsTable('Gold.gold_sales_orders')